In [ ]:
"""
AgroAlert Ghana -- Extended Weather Collection (2019-01-01 to 2023-12-31)
===========================================================================
Same logic as original 03_weather_data.ipynb (final cell), extended date range.
Source: Open-Meteo ERA5 archive API (no auth required, no quota concern --
ERA5 reanalysis covers back to 1940, so this is the easiest of the three
extensions).
"""

import requests
import pandas as pd
import os
import time

communities = [
    {"name": "Tamale",           "region": "Northern",      "lat": 9.4008,  "lon": -0.8393},
    {"name": "Techiman",         "region": "Bono East",     "lat": 7.5833,  "lon": -1.9333},
    {"name": "Kumasi",           "region": "Ashanti",       "lat": 6.6885,  "lon": -1.6244},
    {"name": "Ho",               "region": "Volta",         "lat": 6.6000,  "lon":  0.4700},
    {"name": "Bolgatanga",       "region": "Upper East",    "lat": 10.7856, "lon": -0.8514},
    {"name": "Wa",               "region": "Upper West",    "lat": 10.0601, "lon": -2.5099},
    {"name": "Sunyani",          "region": "Bono",          "lat": 7.3349,  "lon": -2.3123},
    {"name": "Koforidua",        "region": "Eastern",       "lat": 6.0940,  "lon": -0.2591},
    {"name": "Cape Coast",       "region": "Central",       "lat": 5.1053,  "lon": -1.2466},
    {"name": "Sefwi Wiawso",     "region": "Western North", "lat": 6.2069,  "lon": -2.4856},
    {"name": "Damongo",          "region": "Savannah",      "lat": 9.0833,  "lon": -1.8167},
    {"name": "Nalerigu",         "region": "North East",    "lat": 10.5167, "lon": -0.3667},
    {"name": "Dambai",           "region": "Oti",           "lat": 8.0667,  "lon":  0.1833},
    {"name": "Goaso",            "region": "Ahafo",         "lat": 6.8017,  "lon": -2.5181},
    {"name": "Sekondi-Takoradi", "region": "Western",       "lat": 4.9347,  "lon": -1.7137},
]

START_DATE = '2019-01-01'
END_DATE = '2023-12-31'

all_weather = []

for c in communities:
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={c['lat']}&longitude={c['lon']}"
        f"&start_date={START_DATE}&end_date={END_DATE}"
        f"&daily=precipitation_sum,temperature_2m_max,"
        f"temperature_2m_min,relative_humidity_2m_max,"
        f"et0_fao_evapotranspiration"
        f"&timezone=Africa%2FAccra"
    )
    response = requests.get(url)
    if response.status_code != 200:
        print(f"  ERROR for {c['name']}: HTTP {response.status_code} -- {response.text[:200]}")
        continue

    data = response.json()
    if 'daily' not in data:
        print(f"  ERROR for {c['name']}: unexpected response -- {data}")
        continue

    daily = data['daily']
    for i in range(len(daily['time'])):
        all_weather.append({
            'community': c['name'],
            'region': c['region'],
            'date': daily['time'][i],
            'rainfall_mm': daily['precipitation_sum'][i],
            'temp_max': daily['temperature_2m_max'][i],
            'temp_min': daily['temperature_2m_min'][i],
            'humidity': daily['relative_humidity_2m_max'][i],
            'et0': daily['et0_fao_evapotranspiration'][i]
        })
    print(f"✓ {c['name']} weather fetched ({len(daily['time'])} days)")
    time.sleep(0.5)  # polite pacing, avoid rate limits on free tier

df_weather = pd.DataFrame(all_weather)
df_weather['date'] = pd.to_datetime(df_weather['date'])

os.makedirs('C:/Users/ELITE/Documents/AGROALERT/data_raw', exist_ok=True)
OUTPUT_PATH = 'C:/Users/ELITE/Documents/AGROALERT/data_raw/weather_raw_2019_2023.csv'
df_weather.to_csv(OUTPUT_PATH, index=False)

print(f"\nTotal records: {len(df_weather)}")
print(f"Saved to: {OUTPUT_PATH}")
print(f"Communities: {df_weather['community'].nunique()} / {len(communities)}")
print(f"Date range: {df_weather['date'].min()} to {df_weather['date'].max()}")
print(df_weather.head())

print("\n--- IMPORTANT ---")
print("This pull already spans the full 2019-2023 window, so replace (don't merge)")
print("your existing weather_raw.csv with this file to avoid duplicate rows.")
